# 1번 문제 — 현재 작업 실행본

스냅샷 시각: **2026-10-10 06:14:41 UTC**. 최종 제출 확정본이 아니라, 이 시점의 작업 결과를 온라인 Jupyter에서 확인하기 위한 실행본입니다.

이 파일을 **해당 문항의 원본 Notebook과 같은 폴더**에 업로드하고 **Kernel → Restart & Run All**을 실행하세요.
기존 `data/` 폴더를 사용하며, 마지막 셀에서 현재 작업 폴더의 **`submission.csv`**를 생성합니다.
기존 `submission.csv`가 있으면 이 결과로 갱신합니다. 문항마다 별도 폴더에서 실행하세요.
데이터 위치가 다르면 첫 코드 셀의 `DATA_ROOT`만 수정하세요. 추가 산출물 파일 업로드나 패키지 설치는 필요 없습니다.

**실행 내용:** GPU에서 학습·추론을 마친 TabICL 후보 4개의 확률을 저장된 가중치 [0.25, 0.25, 0.25, 0.25]로 합산합니다. 현재 `submission.finetuned.csv`를 재현하며, 입력 데이터와 ID, 확률, 레이블, 정렬을 확인합니다. GPU 학습과 모델 추론 자체는 다시 실행하지 않습니다.

실행 대상은 사용자 제공 온라인 환경(Python 3.9.7, CPU 2개, RAM 4 GiB)입니다.
**온라인 동일 환경 검증 미완료:** 저장된 출력은 로컬 Python 3.12 실행 결과입니다.
Python 3.9 문법 검사와 로컬 전체 실행은 실제 온라인 커널 실행 확인과 구분합니다.
아래 원래 문제 지문 뒤에 실행 셀이 있습니다.

# 레이더 시계열 데이터 기반 이동 표적 식별하기

본 임무의 목적은 **짧은 시간 동안 수집된 레이더 시계열 데이터를 분석하여 이동 표적의 클래스를 식별하는 분류기**를 만드는 것입니다.

가상의 기지 주변 도로와 진입로에는 일반 차량, 대형 차량, 이륜 이동수단, 보행자 등 다양한 이동 표적이 존재합니다. 레이더 센서는 하나의 이동 표적을 관측하는 동안 거리, 방위각, 반사 강도(RCS), 속도, 위치 등의 정보를 여러 시점에 걸쳐 수집합니다.

본 과제에서는 약 **500ms 동안 관측된 하나의 이동 표적**을 하나의 `sample_id`로 정의합니다. 동일한 `sample_id`에는 시간에 따라 수집된 여러 개의 레이더 detection 레코드가 포함될 수 있습니다.

주어진 훈련 데이터를 이용하여 분류 모델을 학습한 뒤, 검증 데이터의 각 `sample_id`가 아래 5개 클래스 중 어디에 해당하는지 예측해야 합니다.

| 레이블 | 설명 |
| --- | --- |
| `LIGHT_VEHICLE` | 일반 승용차급 차량 |
| `HEAVY_VEHICLE` | 대형 차량 |
| `TWO_WHEELER` | 자전거 또는 오토바이류 |
| `PERSONNEL` | 단독 보행자 |
| `PERSONNEL_GROUP` | 여러 명의 보행자 |

평가는 검증 데이터의 각 `sample_id`에 대해 예측한 이동 표적 클래스가 실제 정답과 일치하는지를 기준으로 **Accuracy(정확도)** 를 계산하여 진행합니다.

## 훈련 데이터

`train.csv`에는 총 **1,000개의 이동 표적 sample**이 제공됩니다. 5개 클래스별로 200개의 sample이 포함되어 있습니다.

하나의 sample은 여러 개의 레이더 detection 레코드로 구성되므로, `train.csv`에서 하나의 `sample_id`가 여러 행에 걸쳐 나타날 수 있습니다.

| 필드 | 타입 | 설명 |
| --- | --- | --- |
| `sample_id` | 문자열 | 하나의 이동 표적 관측 구간을 식별하는 ID |
| `t_ms` | 실수 | 해당 sample의 첫 관측 시점으로부터 경과한 시간(ms) |
| `sensor_id` | 숫자 | detection을 수집한 레이더 센서 ID |
| `range_m` | 실수 | detection을 관측한 레이더 센서로부터 표적까지의 거리(m) |
| `azimuth_rad` | 실수 | detection을 관측한 레이더 센서의 로컬 좌표계를 기준으로 측정한 방위각(rad) |
| `rcs_dbsm` | 실수 | Radar Cross Section(dBsm). 레이더 반사 강도와 관련된 값 |
| `radial_velocity_mps` | 실수 | 레이더가 측정한 방사 방향 속도(m/s) |
| `radial_velocity_comp_mps` | 실수 | 센서 이동 영향을 보정한 방사 방향 속도(m/s) |
| `x_m` | 실수 | 센서의 장착 위치와 방향을 반영하여 공통 기준 좌표계로 변환한 detection의 x 위치((m) |
| `y_m` | 실수 | 센서의 장착 위치와 방향을 반영하여 공통 기준 좌표계로 변환한 detection의 y 위치((m) |
| `label` | 문자열 | 예측 대상이 되는 이동 표적 클래스 |

range_m 과 azimuth_rad 는 detection을 수집한 개별 레이더 센서의 로컬 좌표계를 기준으로 측정된 값입니다. 반면 X_m 과 y_m는 각 센서의 장착 위치와 방향을 반 영하여 공통 기준 좌표계로 변환한 위치입니다. 따라서 일반적으로 range_m은 Sqrt(X_m2 + y_m2) 와 직접 일치하지 않을 수 있습니다.

예를 들어 동일한 `sample_id`에 다음과 같이 여러 시점의 detection이 존재할 수 있습니다.

| sample_id | t_ms | sensor_id | range_m | azimuth_rad | rcs_dbsm | radial_velocity_mps | x_m | y_m | label |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| TR_000001 | 0.0 | 1 | ... | ... | ... | ... | ... | ... | LIGHT_VEHICLE |
| TR_000001 | 63.2 | 1 | ... | ... | ... | ... | ... | ... | LIGHT_VEHICLE |
| TR_000001 | 127.5 | 2 | ... | ... | ... | ... | ... | ... | LIGHT_VEHICLE |
| TR_000002 | 0.0 | 3 | ... | ... | ... | ... | ... | ... | PERSONNEL |

위 값은 **파일 구조를 설명하기 위한 예시**이며 실제 데이터 값은 `train.csv`를 사용해야 합니다.

행 하나를 독립적으로 분류하는 것이 아니라, **같은 `sample_id`에 속한 여러 시점의 레이더 정보를 함께 활용하여 하나의 표적 클래스를 예측하는 것**이 핵심입니다.

이 데이터셋을 이용하여 이동 표적 분류 모델을 학습합니다.

## 검증 데이터

`test.csv`에는 총 **300개의 이동 표적 sample**이 포함되어 있습니다.

훈련 데이터와 동일한 레이더 필드가 제공되지만 정답인 `label` 필드는 포함되지 않습니다.

| 필드 | 타입 | 설명 |
| --- | --- | --- |
| `sample_id` | 문자열 | 분류해야 할 이동 표적 sample ID |
| `t_ms` | 실수 | 해당 sample의 첫 관측 시점으로부터 경과한 시간(ms) |
| `sensor_id` | 숫자 | detection을 수집한 레이더 센서 ID |
| `range_m` | 실수 | detection을 관측한 레이더 센서로부터 표적까지의 거리(m) |
| `azimuth_rad` | 실수 | detection을 관측한 레이더 센서의 로컬 좌표계를 기준으로 측정한 방위각(rad) |
| `rcs_dbsm` | 실수 | Radar Cross Section(dBsm) |
| `radial_velocity_mps` | 실수 | 레이더가 측정한 방사 방향 속도(m/s) |
| `radial_velocity_comp_mps` | 실수 | 센서 이동 영향을 보정한 방사 방향 속도(m/s) |
| `x_m` | 실수 | 센서의 장착 위치와 방향을 반영하여 공통 기준 좌표계로 변환한 detection의 x 위치(m) |
| `y_m` | 실수 | 센서의 장착 위치와 방향을 반영하여 공통 기준 좌표계로 변환한 detection의 y 위치(m) |

훈련 데이터로 만든 모델을 이용하여 `test.csv`의 각 `sample_id`에 대해 하나의 이동 표적 클래스를 예측하고, 아래에서 설명하는 제출 파일 형식에 맞춰 결과를 저장하면 됩니다.

## 제출 파일 포맷

`test.csv`의 각 `sample_id`에 대해 하나의 예측 결과를 생성하고 아래와 같은 형태의 CSV 파일로 저장합니다. 파일 이름은 `submission.csv`로 지정합니다.

각 `sample_id`는 결과 파일에 정확히 한 번만 나타나야 하며, 결과는 sample_id를 기준으로 오름차순 정렬합니다.

| sample_id | label |
| --- | --- |
| VA_000001 | LIGHT_VEHICLE |
| VA_000002 | PERSONNEL |
| VA_000003 | TWO_WHEELER |
| VA_000004 | HEAVY_VEHICLE |
| VA_000005 | PERSONNEL_GROUP |

위 결과는 제출 포맷을 보여주기 위한 예시이며 실제 정답과는 관계가 없습니다.

- `sample_id`: `test.csv`에 주어진 ID 값을 그대로 사용합니다.
- `label`: 다음 5개 값 중 하나를 사용합니다.
  - `LIGHT_VEHICLE`
  - `HEAVY_VEHICLE`
  - `TWO_WHEELER`
  - `PERSONNEL`
  - `PERSONNEL_GROUP`
- `test.csv`에 존재하는 모든 `sample_id`에 대해 결과가 있어야 합니다.
- 동일한 `sample_id`가 중복으로 존재하면 안 됩니다.



## 평가 기준

검증 데이터의 각 `sample_id`에 대해 예측 결과와 실제 정답이 일치하는지를 비교하여 **Accuracy(정확도)** 를 계산합니다.

* correct = 이동 표적 클래스를 정확하게 예측한 sample 수
* total = 전체 validation sample 수
* accuracy = correct / total

예를 들어 300개의 validation sample 중 225개를 정확하게 분류했다면:

`Accuracy = 225 / 300 = 0.75`

따라서 점수는 **75.0점**입니다.

각 `sample_id`는 내부에 포함된 detection 레코드 수와 관계없이 동일하게 **1개의 평가 케이스**로 계산됩니다.

## 실행 1. 환경과 경로

In [1]:
import os
for _thread_variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_variable] = "2"

import csv
import hashlib
import json
import platform
import sys
import time
from pathlib import Path
from importlib import metadata
import numpy as np
import pandas as pd
from IPython.display import display, FileLink

# 보통 수정할 필요가 없습니다. 자동 탐색 실패 시 data 폴더 경로를 직접 지정하세요.
DATA_ROOT = None  # 예: Path("./data")
OUTPUT_PATH = Path.cwd() / "submission.csv"
started_at = time.perf_counter()
print("Python:", platform.python_version(), "|", sys.executable)
print("작업 폴더:", Path.cwd())
print("NumPy:", np.__version__, "| pandas:", pd.__version__)
for _package in ("torch", "scikit-learn"):
    try:
        print(_package + ":", metadata.version(_package))
    except metadata.PackageNotFoundError:
        print(_package + ": 미설치 (이 실행본에서 필수 아님)")
print("추가 패키지 설치나 GPU 연결 없이 실행합니다.")

Python: 3.12.3 | /config/workspace/.venv/bin/python
작업 폴더: /config/workspace/race90/notebook-checks/current/Problem #1
NumPy: 2.5.3 | pandas: 3.0.6
torch: 2.14.1+cpu
scikit-learn: 1.9.1
추가 패키지 설치나 GPU 연결 없이 실행합니다.


## 실행 2. 내장 예측 산출물 및 원본 데이터 확인

In [2]:
# 이 데이터는 현재 작업에서 얻은 예측·검토 산출물입니다. 모델 가중치나 정답 데이터가 아닙니다.
import base64
import zlib

_packed_snapshot = (
    '...'
)
_snapshot_bytes = zlib.decompress(base64.b64decode(_packed_snapshot))
assert hashlib.sha256(_snapshot_bytes).hexdigest() == '806d4221ddb337c8388a74d82fb3dfb9c8eb1ca885f4cbd7a1be3bbd46e488ff'
snapshot = json.loads(_snapshot_bytes.decode('utf-8'))
del _packed_snapshot, _snapshot_bytes
print('산출물 스냅샷:', snapshot['created_at_utc'])
print('실행 방식:', snapshot['execution_mode'])

산출물 스냅샷: 2026-10-10 06:14:41 UTC
실행 방식: GPU 저장 확률을 읽어 앙상블·검증·CSV 저장 (모델 재추론 아님)


In [3]:
QUESTION = 1

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def sha256_csv_text(path):
    # UTF-8 BOM과 줄바꿈 방식의 차이는 허용합니다.
    value = Path(path).read_bytes().decode("utf-8-sig").replace("\r\n", "\n")
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

def data_candidates():
    if DATA_ROOT is not None:
        return [Path(DATA_ROOT).expanduser()]
    here = Path.cwd()
    return [here / "data", here,
            here / ("Problem #" + str(QUESTION)) / "data",
            here / ("q" + str(QUESTION)) / "data",
            here / "race90" / "competition" / ("q" + str(QUESTION)) / "data"]

def save_submission(frame):
    if frame.isna().any().any():
        raise ValueError("제출 결과에 결측값이 있습니다.")
    temporary = OUTPUT_PATH.with_name(OUTPUT_PATH.name + ".tmp")
    frame.to_csv(temporary, index=False, encoding="utf-8", lineterminator="\n")
    written = pd.read_csv(temporary, encoding="utf-8")
    pd.testing.assert_frame_equal(written, frame.reset_index(drop=True), check_dtype=False)
    temporary.replace(OUTPUT_PATH)
    print("저장 완료:", OUTPUT_PATH)
    print("행 수:", len(frame), "| 열:", list(frame.columns))
    print("SHA256:", sha256_file(OUTPUT_PATH))
    print("실행 시간: %.2f초" % (time.perf_counter() - started_at))
    display(frame.head(10))
    display(FileLink(str(OUTPUT_PATH.relative_to(Path.cwd()))))

REQUIRED_TRAIN = ['sample_id', 'label', 't_ms']
REQUIRED_TEST = ['sample_id', 't_ms']
data_root = None
for candidate in data_candidates():
    if not all((candidate / name).is_file() for name in ("train.csv", "test.csv")):
        continue
    train_columns = pd.read_csv(candidate / "train.csv", nrows=0, encoding="utf-8-sig").columns
    test_columns = pd.read_csv(candidate / "test.csv", nrows=0, encoding="utf-8-sig").columns
    if set(REQUIRED_TRAIN).issubset(train_columns) and set(REQUIRED_TEST).issubset(test_columns):
        data_root = candidate.resolve()
        break
if data_root is None:
    raise FileNotFoundError("이 문항의 train.csv/test.csv를 찾지 못했습니다. 노트북을 해당 문제 폴더에 넣거나 DATA_ROOT를 수정하세요.")
for name, expected_hash in snapshot["input_csv_sha256"].items():
    if sha256_csv_text(data_root / name) != expected_hash:
        raise ValueError(name + " 내용이 현재 작업 스냅샷과 다릅니다. 올바른 문제 데이터인지 확인하세요.")
train = pd.read_csv(data_root / "train.csv", encoding="utf-8-sig")
test = pd.read_csv(data_root / "test.csv", encoding="utf-8-sig")
print("입력 확인:", data_root, "| train", train.shape, "| test", test.shape)

입력 확인: /config/workspace/race90/competition/q1/data | train (35776, 11) | test (10729, 10)


## 실행 3. 확률 앙상블과 개발 검증

개발 OOF 정확도는 훈련 데이터에 대한 교차검증 지표이며 실제 테스트 점수가 아닙니다. 후보 선택에도 사용한 값이므로 최종 일반화 성능을 보장하지 않습니다.

In [4]:
allowed = {"LIGHT_VEHICLE", "HEAVY_VEHICLE", "TWO_WHEELER", "PERSONNEL", "PERSONNEL_GROUP"}
classes = np.asarray(snapshot["members"][0]["classes"])
if len(classes) != 5 or set(classes) != allowed:
    raise ValueError("클래스 목록 오류")
ids = snapshot["test_ids"]
if len(ids) != 300 or len(set(ids)) != 300 or sorted(test["sample_id"].unique()) != sorted(ids):
    raise ValueError("테스트 sample_id 불일치")
weights = np.asarray(snapshot["weights"], dtype=float)
if len(weights) != len(snapshot["members"]) or np.any(weights < 0) or not np.isclose(weights.sum(), 1.0):
    raise ValueError("앙상블 가중치 오류")
probabilities = np.zeros((300, 5), dtype=float)
oof_probabilities = np.zeros((len(snapshot["train_ids"]), 5), dtype=float)
for weight, member in zip(weights, snapshot["members"]):
    if member["classes"] != classes.tolist():
        raise ValueError("모델별 클래스 순서 불일치")
    for key, accumulator in (("test", probabilities), ("oof", oof_probabilities)):
        values = np.asarray(member[key], dtype=float)
        if values.shape != accumulator.shape or not np.isfinite(values).all() or (values < 0).any():
            raise ValueError("예측 확률 배열 오류: " + key)
        if not np.allclose(values.sum(axis=1), 1.0, atol=1e-4):
            raise ValueError("확률 합 오류: " + key)
        accumulator += weight * values
df = pd.DataFrame({"sample_id": ids, "label": classes[probabilities.argmax(axis=1)]})
df = df.sort_values("sample_id").reset_index(drop=True)
expected = pd.DataFrame(snapshot["reference_predictions"]).sort_values("sample_id").reset_index(drop=True)
pd.testing.assert_frame_equal(df, expected)
if train.groupby("sample_id")["label"].nunique().ne(1).any():
    raise ValueError("같은 훈련 sample_id에 여러 레이블이 있습니다.")
truth = train.groupby("sample_id")["label"].first().reindex(snapshot["train_ids"])
if truth.isna().any():
    raise ValueError("OOF 검증용 훈련 ID 불일치")
oof_accuracy = float(np.mean(truth.to_numpy() == classes[oof_probabilities.argmax(axis=1)]))
if not np.isclose(oof_accuracy, snapshot["validation"]["accuracy"]):
    raise ValueError("저장된 개발 검증 지표와 불일치")
print("현재 예측 300건과 모두 일치")
print("개발 OOF Accuracy: %.4f (테스트 정답 점수 아님)" % oof_accuracy)
display(df["label"].value_counts().rename("예측 수"))

현재 예측 300건과 모두 일치
개발 OOF Accuracy: 0.8880 (테스트 정답 점수 아님)


label
PERSONNEL_GROUP    73
PERSONNEL          61
HEAVY_VEHICLE      57
LIGHT_VEHICLE      55
TWO_WHEELER        54
Name: 예측 수, dtype: int64

## 실행 4. 제출 CSV 저장

In [5]:
assert list(df.columns) == ["sample_id", "label"]
assert len(df) == 300 and df["sample_id"].is_unique
assert df["sample_id"].tolist() == sorted(test["sample_id"].unique())
assert set(df["label"]).issubset(allowed)
save_submission(df)

저장 완료: /config/workspace/race90/notebook-checks/current/Problem #1/submission.csv
행 수: 300 | 열: ['sample_id', 'label']
SHA256: 7f7261489bdaf9a457d5b08db3b01bd381340c1b44e2a7ce0923fe5ec9c63e2e
실행 시간: 0.49초


,sample_id,label
0,VA_000001,PERSONNEL_GROUP
1,VA_000002,TWO_WHEELER
2,VA_000003,LIGHT_VEHICLE
3,VA_000004,HEAVY_VEHICLE
4,VA_000005,TWO_WHEELER
5,VA_000006,TWO_WHEELER
6,VA_000007,PERSONNEL_GROUP
7,VA_000008,TWO_WHEELER
8,VA_000009,PERSONNEL_GROUP
9,VA_000010,PERSONNEL


/config/workspace/race90/notebook-checks/current/Problem #1/submission.csv

## 참고: 현재 GPU 작업 소스

아래 소스는 이미 수행한 GPU 계산의 기록입니다. Markdown이므로 Run All에서 실행하지 않습니다. 온라인에서 모델을 재학습·재추론한 결과로 해석하지 않습니다.

<details><summary>sample_id별 기본 특징 추출</summary>

```python
def radar_features(data):
    rows = []
    for sid, g in data.groupby('sample_id', sort=True):
        g = g.sort_values('t_ms')
        f = {'sample_id': sid, 'count': len(g), 'times': g.t_ms.nunique(),
             'sensors': g.sensor_id.nunique()}
        cols = ['rcs_dbsm', 'radial_velocity_comp_mps', 'radial_velocity_mps',
                'range_m', 'azimuth_rad', 'x_m', 'y_m']
        for c in cols:
            v = g[c].to_numpy()
            for name, value in [('mean', v.mean()), ('std', v.std()), ('min',v.min()),
                                ('max',v.max()), ('q10',np.quantile(v,.1)),
                                ('median',np.median(v)), ('q90',np.quantile(v,.9))]:
                f[c+'_'+name] = value
        t = g.t_ms.to_numpy()/1000
        tc = t-t.mean()
        denom = np.dot(tc,tc)+1e-12
        residuals=[]
        for c in ['x_m','y_m','rcs_dbsm','radial_velocity_comp_mps']:
            v=g[c].to_numpy(); slope=np.dot(tc,v-v.mean())/denom
            residual=v-v.mean()-slope*tc
            f[c+'_slope']=slope; f[c+'_resid_std']=residual.std()
            f[c+'_resid_qspan']=np.quantile(residual,.9)-np.quantile(residual,.1)
            if c in ['x_m','y_m']: residuals.append(residual)
        eig=np.linalg.eigvalsh(np.cov(residuals))
        f['spatial_eigen_small'],f['spatial_eigen_large']=eig
        f['track_speed']=np.hypot(f['x_m_slope'],f['y_m_slope'])
        f['comp_abs_mean']=g.radial_velocity_comp_mps.abs().mean()
        counts=g.groupby('t_ms').size().to_numpy()
        f['detections_per_time_mean']=counts.mean();f['detections_per_time_std']=counts.std()
        for c in ['x_m','y_m','rcs_dbsm','radial_velocity_comp_mps']:
            s=g.groupby('t_ms')[c].std().fillna(0)
            f[c+'_within_time_std']=s.mean()
        for sensor in [1,2,3,4]:
            sg=g[g.sensor_id==sensor]
            f['sensor_'+str(sensor)+'_fraction']=len(sg)/len(g)
        rows.append(f)
    return pd.DataFrame(rows).set_index('sample_id').fillna(0)
```

</details>

<details><summary>추가 특징 추출</summary>

```python
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from custom_adapters.competition import radar_features


def sensor_geometry(data):
    transforms={}
    for sid,g in data.groupby('sensor_id'):
        a=np.c_[g.range_m*np.cos(g.azimuth_rad),g.range_m*np.sin(g.azimuth_rad),np.ones(len(g))]
        transforms[int(sid)]=np.linalg.lstsq(a,g[['x_m','y_m']].values,rcond=None)[0]
    return transforms


def features(data,geometry):
    basic=radar_features(data);rows=[]
    for sid,g in data.groupby('sample_id',sort=True):
        g=g.sort_values('t_ms');n=len(g);f={'sample_id':sid}
        t=g.t_ms.to_numpy()/1000;t=t-t.mean()
        pos=g[['x_m','y_m']].to_numpy();pos=pos-pos.mean(0)
        v=g.radial_velocity_comp_mps.to_numpy();r=g.rcs_dbsm.to_numpy()
        directions=[]
        for a,sen in zip(g.azimuth_rad,g.sensor_id):
            rot=geometry[int(sen)][:2,:]
            directions.append(np.array([np.cos(a),np.sin(a)])@rot)
        u=np.array(directions)
        velocity=np.linalg.lstsq(u,v,rcond=None)[0]
        f['doppler_speed']=np.linalg.norm(velocity)
        f['doppler_vx'],f['doppler_vy']=velocity
        f['doppler_resid_std']=np.std(v-u@velocity)
        f['doppler_resid_mad']=np.median(np.abs(v-u@velocity-np.median(v-u@velocity)))
        f['direction_condition']=np.linalg.cond(u.T@u)
        for ridge in [.01,.1,1,10]:
            vel=np.linalg.solve(u.T@u+ridge*np.eye(2),u.T@v)
            f['speed_ridge_'+str(ridge)]=np.linalg.norm(vel)
        # Joint least squares: positions at different times and radial velocities.
        for w in [.1,1,10]:
            vel=np.linalg.solve(u.T@u+w*np.dot(t,t)*np.eye(2),u.T@v+w*(t[:,None]*pos).sum(0))
            f['speed_joint_'+str(w)]=np.linalg.norm(vel)
        temporal_vel=(t[:,None]*pos).sum(0)/(np.dot(t,t)+1e-12)
        for mode,vel in [('raw',np.zeros(2)),('time',temporal_vel),('doppler',velocity)]:
            p=pos-t[:,None]*vel
            cov=p.T@p/max(n-1,1);e,rot=np.linalg.eigh(cov);pca=p@rot
            for i in range(2):
                f[mode+'_eig'+str(i)]=e[i]
                f[mode+'_pca_skew'+str(i)]=skew(pca[:,i])
                f[mode+'_pca_kurt'+str(i)]=kurtosis(pca[:,i])
                for q in [0.8,0.9,0.95,1.0]:
                    f[mode+'_span'+str(i)+'_'+str(q)]=np.quantile(pca[:,i],q)-np.quantile(pca[:,i],1-q)
            f[mode+'_xy_cov']=cov[0,1]
            f[mode+'_area']=np.sqrt(max(np.linalg.det(cov),0))
            f[mode+'_ratio']=e[1]/max(e[0],1e-8)
            f[mode+'_xvar']=cov[0,0];f[mode+'_yvar']=cov[1,1]
        within_cov=np.zeros((2,2));dof=0
        for _,gg in g.groupby('t_ms'):
            pp=gg[['x_m','y_m']].to_numpy();pp=pp-pp.mean(0)
            within_cov+=pp.T@pp;dof+=max(len(gg)-1,0)
        within_cov/=max(dof,1);e=np.linalg.eigvalsh(within_cov)
        f['within_eig0'],f['within_eig1']=e
        f['within_xvar']=within_cov[0,0];f['within_yvar']=within_cov[1,1]
        f['within_dof']=dof
        for name,val in [('rcs',r),('velocity',v),('absvelocity',np.abs(v))]:
            f[name+'_skew']=skew(val);f[name+'_kurt']=kurtosis(val)
            f[name+'_iqr']=np.quantile(val,.75)-np.quantile(val,.25)
            f[name+'_mad']=np.median(np.abs(val-np.median(val)))
        rows.append(f)
    extra=pd.DataFrame(rows).set_index('sample_id')
    return pd.concat([basic,extra],axis=1).replace([np.inf,-np.inf],np.nan).fillna(0)


if __name__=='__main__':
    root=Path('competition/q1');train=pd.read_csv(root/'data/train.csv');test=pd.read_csv(root/'data/test.csv')
    geo=sensor_geometry(train)
    X=features(train,geo);Xt=features(test,geo)
    X.to_csv(root/'features_improved.csv');Xt.to_csv(root/'features_test_improved.csv')
    train.groupby('sample_id').label.first().reindex(X.index).to_csv(root/'labels.csv')
    import json
    (root/'sensor_geometry.json').write_text(json.dumps({k:v.tolist() for k,v in geo.items()}))
    print(X.shape,flush=True)
```

</details>

<details><summary>교차검증 및 TabICL 미세조정</summary>

```python
"""Outer-fold evaluation with early stopping restricted to inner training data."""
from pathlib import Path
import argparse, gc, json, time
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, log_loss
from tabicl import FinetunedTabICLClassifier

p = argparse.ArgumentParser()
p.add_argument('--device', default='cuda:0')
p.add_argument('--features', default='original')
p.add_argument('--lr', type=float, default=1e-5)
p.add_argument('--epochs', type=int, default=60)
p.add_argument('--seed', type=int, default=42)
p.add_argument('--cv-seed', type=int, default=42)
p.add_argument('--freeze-col', action='store_true')
p.add_argument('--freeze-row', action='store_true')
p.add_argument('--estimators', type=int, default=32)
p.add_argument('--fine-estimators', type=int, default=4)
p.add_argument('--validation-estimators', type=int, default=8)
p.add_argument('--patience', type=int, default=12)
p.add_argument('--tag', default='')
args = p.parse_args()
torch.set_num_threads(4)
root = Path('/workspace/competition-radar')
X = pd.read_csv(root/'features_physics_train.csv', index_col=0)
Xt = pd.read_csv(root/'features_physics_test.csv', index_col=0)
y = pd.read_csv(root/'labels.csv').label.to_numpy()
counts = {'original':79, 'all':165, 'robust':215, 'physics':337}
columns = list(X.columns[:counts[args.features]])
X, Xt = X[columns], Xt[columns]
name = f'finetune_{args.features}_lr{args.lr:g}_s{args.seed}_cv{args.cv_seed}'
if args.freeze_col: name += '_freeze_col'
if args.freeze_row: name += '_freeze_row'
if args.tag: name += '_'+args.tag
out = np.zeros((len(y),5))
test = np.zeros((len(Xt),5))
scores = []
start = time.time()
for fold,(a,b) in enumerate(StratifiedKFold(5,shuffle=True,random_state=args.cv_seed).split(X,y)):
    model_dir = root/'finetuned_models'/name/str(fold)
    model_dir.mkdir(parents=True,exist_ok=True)
    m = FinetunedTabICLClassifier(
        epochs=args.epochs,learning_rate=args.lr,n_estimators_finetune=args.fine_estimators,
        n_estimators_validation=args.validation_estimators,n_estimators_inference=args.estimators,
        validation_split_ratio=.15,early_stopping=True,patience=args.patience,
        eval_metric='log_loss',device=args.device,random_state=args.seed,
        save_interval=0,verbose=True,freeze_col=args.freeze_col,freeze_row=args.freeze_row,
        extra_classifier_kwargs={'n_jobs':4},
    )
    # Outer validation labels are never used for fitting or early stopping.
    m.fit(X.iloc[a],y[a],output_dir=model_dir)
    out[b] = m.predict_proba(X.iloc[b])
    test += m.predict_proba(Xt)/5
    classes = m.classes_
    scores.append(accuracy_score(y[b],classes[out[b].argmax(1)]))
    print(name,'FOLD',fold,scores[-1],'SECONDS',time.time()-start,flush=True)
    np.savez_compressed(root/(name+'_partial.npz'),oof=out,test=test,classes=classes)
    del m
    gc.collect(); torch.cuda.empty_cache()
np.savez_compressed(root/(name+'.npz'),oof=out,test=test,classes=classes)
result={'args':vars(args),'accuracy':accuracy_score(y,classes[out.argmax(1)]),
        'log_loss':log_loss(y,out),'fold_accuracies':scores,'seconds':time.time()-start,
        'validation':'5 outer folds; fine-tuning early stopping uses only internal split of outer training data.'}
(root/(name+'.json')).write_text(json.dumps(result,indent=2))
print(name,'TOTAL',result,flush=True)
```

</details>